# Notebook 2 — Phase B: Reward-Weighted SFT

Fine-tunes Qwen2.5-1.5B-Instruct with LoRA using per-sample advantage weights.

**Ablation runs** — change `ABLATION` in Cell 2 and re-run from Cell 2:
- `"L1"` — K=4 soft filter (above-mean binary weights)
- `"L2"` — K=4 advantage-weighted
- `"L2_pf"` — same data built with prefix-feedback ON

**Baseline** — set `ABLATION = "baseline"` and ensure `sft_data_baseline.jsonl` was built
by filtering L1 to top-1 correct continuation per group with `weight=1.0`.

In [ ]:
# Cell 1 — Mount Drive & Install
from google.colab import drive
drive.mount('/content/drive')

# torchao 0.10.0 ships with Colab's PyTorch but peft requires >=0.16.0.
# We don't use torchao, so uninstall it to avoid the ImportError in peft.
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q transformers accelerate peft trl bitsandbytes

import os
BASE_DIR  = "/content/drive/MyDrive/P-ALIGN"
MODEL_DIR = f"{BASE_DIR}/models/Qwen2.5-1.5B-Instruct"
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# Cell 2 — Config Switch
# ── CHANGE THIS FOR EACH ABLATION RUN ──────────────────────────────────────
ABLATION  = "L2"           # "L1" | "L2" | "L2_pf" | "baseline"
DATA_FILE = f"{BASE_DIR}/data/sft_data_{ABLATION}.jsonl"
CKPT_DIR  = f"{BASE_DIR}/checkpoints/sft_{ABLATION}"
# ───────────────────────────────────────────────────────────────────────────
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Training {ABLATION} → {CKPT_DIR}")

In [ ]:
# Cell 3 — Load & Tokenize Dataset
import json, torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
MAX_LEN = 2048

class WeightedSFTDataset(Dataset):
    def __init__(self, path: str):
        with open(path) as f:
            self.rows = [json.loads(l) for l in f if l.strip()]

    def __len__(self): return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        messages = [
            {"role": "user",      "content": f"Question: {row['question']}"},
            {"role": "assistant", "content": row["full_sequence"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        enc = tokenizer(text, max_length=MAX_LEN, truncation=True,
                        padding="max_length", return_tensors="pt")
        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        prompt_only = tokenizer.apply_chat_template(
            [{"role": "user", "content": f"Question: {row['question']}"}],
            tokenize=False, add_generation_prompt=True
        )
        prompt_len = len(tokenizer(prompt_only, return_tensors="pt")["input_ids"][0])
        labels[:prompt_len]         = -100
        labels[attention_mask == 0] = -100

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels,
            "weight":         torch.tensor(row["weight"], dtype=torch.float32),
        }

import os
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"Data file not found: {DATA_FILE}\n"
        "Run nb01_phase_a_data_building.ipynb first to generate the SFT data."
    )

dataset = WeightedSFTDataset(DATA_FILE)

if len(dataset) == 0:
    raise ValueError(
        f"Dataset is empty: {DATA_FILE}\n"
        "The file exists but contains no valid rows. "
        "Check that nb01 completed successfully (stats: all_wrong groups are skipped)."
    )

print(f"Dataset size: {len(dataset)}")
# Sanity-check first sample weights are non-zero
sample = dataset[0]
print(f"  weight[0]={sample['weight'].item():.4f}  "
      f"  label tokens={(sample['labels'] != -100).sum().item()}")

In [ ]:
# Cell 4 — Data Collator
from dataclasses import dataclass
from typing import Any

@dataclass
class WeightedDataCollator:
    def __call__(self, features: list) -> dict:
        keys  = ["input_ids", "attention_mask", "labels"]
        batch = {k: torch.stack([f[k] for f in features]) for k in keys}
        batch["weights"] = torch.stack([f["weight"] for f in features])
        return batch

In [ ]:
# Cell 5 — Custom Trainer (Weighted NLL Loss)
from transformers import Trainer
import torch.nn.functional as F

class WeightedSFTTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        weights = inputs.pop("weights")          # [B]
        outputs = model(**inputs)
        logits  = outputs.logits                 # [B, T, V]
        labels  = inputs["labels"]               # [B, T]
        B, T, V = logits.shape

        shift_logits = logits[:, :-1, :].contiguous().view(-1, V)
        shift_labels = labels[:, 1:].contiguous().view(-1)
        token_loss   = F.cross_entropy(shift_logits, shift_labels, reduction="none")
        token_loss   = token_loss.view(B, T - 1)

        valid      = (labels[:, 1:] != -100).float()
        per_sample = (token_loss * valid).sum(-1) / (valid.sum(-1) + 1e-6)

        w = weights.to(per_sample.device)
        loss = (per_sample * w).sum() / (w.sum() + 1e-6)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Cell 6 — Load Model + LoRA
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map="auto"
)
base_model.config.use_cache = False  # required for gradient checkpointing

lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
model.enable_input_require_grads()

In [ ]:
# Cell 7 — Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,   # effective batch = 32
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    dataloader_num_workers=0,
    report_to="none",
    remove_unused_columns=False,     # keep "weights" column
)

In [ ]:
# Cell 8 — Train
trainer = WeightedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=WeightedDataCollator(),
)
trainer.train()
trainer.save_model(CKPT_DIR)
tokenizer.save_pretrained(CKPT_DIR)
print(f"Saved to {CKPT_DIR}")